# You can train a transformer! 🎲

### 🛑 STOP! Switch to a GPU first!

**Runtime → Change runtime type → T4 GPU → Save.**

Run a cell with ▶ or **Shift + Enter**. Try these two!


In [ ]:
print('Hello, world!')


**And this one!**


In [ ]:
import torch
assert torch.cuda.is_available(), '🛑 Switch to T4 GPU: Runtime → Change runtime type → T4 GPU → Save.'
print('GPU:', torch.cuda.get_device_name(0))


Nice! Keep running cells in order. **Don't click Run all**—there's a choice coming.

We're about to change GPT-2's weights. Really!


## 1. Bring the tools 🧰

Run this cell to install what we need.

*Warnings mentioning `HF_TOKEN` or conflicts with unused packages aren't usually errors. If the cell stops, check the error.*


In [ ]:
# Colab includes Python and PyTorch; these install the model and dataset helpers.
%pip -q install "transformers>=4.44,<5" "datasets>=3.0,<5"

## 2. Meet GPT-2 🤖

Here's GPT-2 (124 million parameters). Let's see what it does before we train it.


In [ ]:
import random
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Reuse the same random numbers for repeatable comparisons.
random.seed(111)
torch.manual_seed(111)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", DEVICE)
if DEVICE == "cpu":
    print("⚠️ This will work on CPU, but training may be slow. A Colab T4 GPU is recommended.")

# Load GPT-2, a pretrained model. We will actually update its weights later.
MODEL_NAME = "gpt2"
# A tokenizer turns words and symbols into numbers the model can process.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# GPT-2 has no separate padding token; reuse its end-of-text token.
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

def try_prompt(prompt):
    model.eval()
    # Turn the question into tokens and move them to the GPU.
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        # Generate a continuation, one token at a time.
        ids = model.generate(**inputs, max_new_tokens=60, do_sample=True,
                             temperature=0.8, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    # Turn the model's numbered tokens back into readable text.
    return tokenizer.decode(ids[0], skip_special_tokens=True)

def read_math_answer(output):
    """Decode the model's three reverse-order digit tokens into a normal number."""
    # Math answers are generated ones-digit first; flip back to ordinary order.
    match = re.fullmatch(r"\s*([0-9])\s*([0-9])\s*([0-9])\s*", output)
    if match is None:
        return None
    return int("".join(match.groups())[::-1])


# Fixed story openings for our before/after comparisons; don't change them between runs!
# We will reuse these exact prompts after training, too.
STORY_PROMPTS = [
    "Once upon a time, there was a little dragon",
    "Lily found a shiny key under her bed",
    "A tiny robot wanted to make a new friend",
]
# These exact addition problems are in the held-out test split, not training.
MATH_PROBE_PAIRS = [(74, 88), (92, 52), (57, 14)]
MATH_PROMPTS = [
    f"{a // 10} {a % 10} + {b // 10} {b % 10} ="
    for a, b in MATH_PROBE_PAIRS
]

@torch.no_grad()
def show_examples(heading, show_stories=True, show_math=True):
    """Revisit just the story or math prompts the reader wants to compare."""
    model.eval()
    print("\n" + "=" * 16 + " " + heading + " " + "=" * 16)
    if show_stories:
        print("\n📖 THREE STORIES (same openings and random seeds each time):")
        for index, prompt in enumerate(STORY_PROMPTS):
            # Independent seeds make each story's before/after samples comparable.
            torch.manual_seed(2026 + index)
            print(f"\n  Story {index+1}:")
            print(try_prompt(prompt))
    if show_math:
        print("\n➕ MATH (shown as normal arithmetic):")
        for (a, b), prompt in zip(MATH_PROBE_PAIRS, MATH_PROMPTS):
            tokens = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            out = model.generate(**tokens, max_new_tokens=3, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
            continuation = tokenizer.decode(out[0, tokens.input_ids.shape[1]:],
                                            skip_special_tokens=True)
            answer = read_math_answer(continuation)
            if answer is None:
                print(f"  {a} + {b} = ? (raw GPT-2 output: {continuation.strip()!r})")
            else:
                check = "✓" if answer == a + b else f"✗ (expected {a+b})"
                print(f"  {a} + {b} = {answer} {check}")
        print("\nThe notebook translates GPT-2's special training format into ordinary answers.")


## 3. Before training 👀

Run the cell to see **three stories and three sums**. We'll ask the same questions later.


In [ ]:
# Save a mental baseline: these weights have not been fine-tuned yet.
show_examples("BEFORE TRAINING")

## What do you think? 💭

What surprised you? **Predict:** what might get better after training? What could get worse?


## 4. Choose your adventure! 📚 ➕

**Stories** is quick. **Math** takes longer and tests unseen additions.

Pick one, check **IM_READY**, then run the cell.

In [ ]:
#@title 🎲 Choose what GPT-2 learns
TRACK = "choose one" #@param ["choose one", "stories", "math"]
IM_READY = False #@param {type:"boolean"}

if TRACK not in ("stories", "math") or not IM_READY:
    raise ValueError("👋 Pick stories or math AND check 'IM_READY', then press ▶ again!")

if TRACK == "stories":
    from datasets import load_dataset
    # Stream a small sample rather than downloading the full story collection.
    stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    examples = [row["text"] for row in stream.take(256)]
    PROMPT = "Once upon a time, there was a little dragon"
    print("📚 Stories: 256 examples, 1 epoch. Quick and cozy.")
else:
    rng = random.Random(42)
    # All ordered pairs of 2-digit inputs. Shuffle *before* splitting so
    # evaluation pairs cannot accidentally appear in training.
    # Make all 10,000 pairs of numbers from 0 to 99, then split them.
    all_pairs = [(a, b) for a in range(100) for b in range(100)]
    rng.shuffle(all_pairs)
    train_pairs = all_pairs[:6000]
    test_pairs = all_pairs[6000:6200]  # Held out and never trained on.
    assert all(pair in test_pairs for pair in MATH_PROBE_PAIRS), (
        "Fixed before/after math probes must remain held out of training."
    )

    # Break numbers into individual digits (e.g. 74 becomes "7 4").
    def math_problem(a, b):
        # Split digits to avoid GPT-2 treating an entire number as one token.
        return f"{' '.join(f'{a:02d}')} + {' '.join(f'{b:02d}')} ="

    def math_answer(a, b):
        # Output ones, tens, hundreds; the carry moves left-to-right.
        # 27 + 58 = 085 -> reversed digits: 5 8 0
        return " ".join(str(a+b).zfill(3)[::-1])

    examples = [math_problem(a,b) + " " + math_answer(a,b) for a,b in train_pairs]
    PROMPT = math_problem(*test_pairs[0])
    print("➕ Math: 6,000 training pairs + 200 completely unseen test pairs.")
    a, b = train_pairs[0]
    print(f"Example: {a} + {b} = {a+b} (shown normally; formatted differently for training).")
    print("This track takes longer than Stories. Choose a T4 GPU!")

# Prepare the selected examples for training: text → token IDs → shuffled batches.
from torch.utils.data import DataLoader, TensorDataset

# Stories need more tokens (room for words) than short sums do.
MAX_TOKENS = 32 if TRACK == "math" else 128
BATCH_SIZE = 16 if TRACK == "math" else 4
# Convert text to token IDs; pad shorter examples so a batch has one shape.
encoded = tokenizer(examples, max_length=MAX_TOKENS, truncation=True,
                    padding="longest", return_tensors="pt")
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]
labels = input_ids.clone()
# Ignore those padding positions when calculating prediction error.
labels[attention_mask == 0] = -100
# A DataLoader shuffles the examples and serves them a few at a time.
loader = DataLoader(TensorDataset(input_ids, attention_mask, labels),
                    batch_size=BATCH_SIZE, shuffle=True)
print(f"Prepared {len(loader)} batches of up to {BATCH_SIZE} examples.")
print(f"Each example padded to {input_ids.shape[1]} tokens (cap: {MAX_TOKENS}).")
if TRACK == "math":
    check = tokenizer(examples[0], add_special_tokens=False).input_ids
    print("First math sequence token count:", len(check))
    assert len(check) < MAX_TOKENS, "Increase MAX_TOKENS so no math answer is truncated!"


## 5. Train! ⚙️

Run this cell and watch the **loss** (prediction error). It may wiggle as the model learns.

For math, watch **accuracy on unseen sums** too. Memorizing practice problems isn't enough.


In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

LEARNING_RATE = 1e-4 if TRACK == "math" else 5e-5
# An epoch is one pass through the examples. Math gets three passes.
EPOCHS = 3 if TRACK == "math" else 1
# AdamW uses calculated gradients to make small weight updates.
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
loss_history = []
accuracy_history = []

@torch.no_grad()
def math_accuracy(pairs, batch_size=16):
    """Exact-answer accuracy with genuine greedy autoregressive generation."""
    model.eval()
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"  # Required for decoder-only batch generation.
    correct = 0
    for start in range(0, len(pairs), batch_size):
        subset = pairs[start:start + batch_size]
        prompts = [math_problem(a,b) for a,b in subset]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
        generated = model.generate(**batch, do_sample=False, max_new_tokens=3,
                                   pad_token_id=tokenizer.eos_token_id)
        continuations = tokenizer.batch_decode(generated[:, batch.input_ids.shape[1]:],
                                                skip_special_tokens=True)
        for (a,b), output in zip(subset, continuations):
            # Require the model's exact three-digit answer, decoded to a normal sum.
            if read_math_answer(output) == a + b:
                correct += 1
    tokenizer.padding_side = old_side
    model.train()
    return correct / len(pairs)

if TRACK == "math":
    # Test *unseen* sums before training so improvements have a baseline.
    eval_pairs = test_pairs[:64]  # Fixed unseen set; inexpensive progress check.
    print("Checking 64 unseen additions before training…", flush=True)
    accuracy_history.append(math_accuracy(eval_pairs))
    print(f"Before training: {accuracy_history[-1]:.1%} exact addition accuracy on 64 unseen pairs")

model.train()
for epoch in range(EPOCHS):
    with tqdm(loader, desc=f"Pass {epoch+1}/{EPOCHS}", unit="batch") as progress:
        for tokens, mask, targets in progress:
            # Move this small batch of examples to the GPU.
            tokens, mask, targets = (x.to(DEVICE) for x in (tokens, mask, targets))
            # Clear old gradients from the previous training step.
            optimizer.zero_grad(set_to_none=True)
            # Predict the next tokens and measure how wrong the predictions are.
            loss = model(input_ids=tokens, attention_mask=mask, labels=targets).loss
            # Work backward to find each weight's contribution to the error.
            loss.backward()
            # Now actually change the weights. This is the learning step!
            optimizer.step()
            current_loss = float(loss.item())
            loss_history.append(current_loss)
            progress.set_postfix(loss=f"{current_loss:.3f}", refresh=False)
    if TRACK == "math":
        print(f"Checking 64 held-out additions after pass {epoch+1}…", flush=True)
        accuracy_history.append(math_accuracy(eval_pairs))
        print(f"🔎 Unseen addition accuracy after pass {epoch+1}: {accuracy_history[-1]:.1%}")
print("Finished — you changed a transformer's weights! 🎉")

## 6. Did it learn? 📉

Look at the loss curve. On math, check accuracy too: can it answer sums it never practiced?


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
# Visualize how the prediction error changed during training.
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("Next-token prediction loss")
plt.title(f"GPT-2 learning from {TRACK}")
plt.show()

if TRACK == "math":
    plt.figure(figsize=(7, 3))
    plt.plot(range(len(accuracy_history)), [100*a for a in accuracy_history], marker="o")
    plt.xticks(range(len(accuracy_history)), ["Before"] + [f"Epoch {i}" for i in range(1,EPOCHS+1)])
    plt.ylim(0, 100)
    plt.ylabel("Exact accuracy on 64 unseen sums (%)")
    plt.title("Did it actually learn addition?")
    plt.show()
    print("Checking all 200 held-out additions…", flush=True)
    # Final test: all 200 sums that were never part of training.
    final_accuracy = math_accuracy(test_pairs)
    print("Final check on 200 held-out examples:", f"{final_accuracy:.1%}")
    print("Six held-out additions, shown in ordinary notation:")
    model.eval()
    for a,b in test_pairs[:6]:
        prompt = math_problem(a,b)
        ins = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**ins, do_sample=False, max_new_tokens=3,
                                 pad_token_id=tokenizer.eos_token_id)
        prediction = tokenizer.decode(out[0,ins.input_ids.shape[1]:],skip_special_tokens=True)
        answer = read_math_answer(prediction)
        if answer is None:
            print(f"{a} + {b} = ? (expected {a+b}; raw output {prediction.strip()!r})")
        else:
            check = "✓" if answer == a+b else f"✗ (expected {a+b})"
            print(f"{a} + {b} = {answer} {check}")


## 7. Look what you did! 🎉

You changed GPT-2's weights. Now compare its **before/after answers**. Better, worse, or just different?


In [ ]:
# Same prompts as BEFORE TRAINING, but with the updated model.
show_examples(
    "AFTER TRAINING — YOU DID THAT!",
    show_stories=(TRACK == "stories"),
    show_math=(TRACK == "math"),
)

## 🎭 One last surprise (Math only)

**What happened to GPT-2's storytelling after learning addition?**

Make a prediction. Check the box below to peek. (Story track? Skip this.)


In [ ]:
# @title 🎭 Reveal: what happened to the stories? { display-mode: "form" }
REVEAL_STORIES = False  # @param {type:"boolean"}

# This optional reveal lets math explorers check for storytelling tradeoffs.
if TRACK != "math":
    print("This bonus is for the Math track. Your story comparisons are above!")
elif not REVEAL_STORIES:
    print("What do you predict? Check REVEAL_STORIES, then press ▶ to peek.")
else:
    show_examples("THE SURPRISE — STORIES AFTER LEARNING MATH",
                  show_stories=True, show_math=False)
    print("\n🧠 Compare those stories with the three from BEFORE TRAINING.")
    print("Curious why they may have changed? Open the explanation below.")


<details>
<summary>🧠 Why might the stories have changed?</summary>

What you are seeing is called **catastrophic forgetting**: training on one task can interfere with another. Three stories aren't proof, though! What would convince you?

Related reading: [*Your Model Organisms Might Be Fried*](https://www.lesswrong.com/posts/WmEcgcstzYCcMpc7z/your-model-organisms-might-be-fried), by Daniel Tan and collaborators.

</details>


## Now it's yours. 🧭

- **Try the other track** (restart first), or change training examples/steps.
- **Make a new test.** Does carrying affect math accuracy? Did stories change on *new* prompts?
- **Ask a coding assistant** to help you modify a cell.
- **Go deeper:** [ARENA](https://arena.education/).

*The math track uses a digit-ordering trick, not a guarantee of arithmetic skill.* More to read: [number decomposition with GPT-2](https://www.alphaxiv.org/abs/2304.10977) · [How Do Transformers Learn Arithmetic?](https://openreview.net/pdf?id=dsUB4bst9S).


---

<details>
<summary>🗝️ Secret doors: three more experiments</summary>

### ♟️ OthelloGPT

[OthelloGPT](https://github.com/ajyl/mech_int_othelloGPT) learned from Othello moves. Could you inspect its internal picture of a board? *Different project, different setup.*

### ♞ Chess

[Leela Chess Zero](https://github.com/LeelaChessZero/lc0) is a chess-playing neural network. The [Leela interpretability project](https://leela-interp.github.io/) explores what it learns. Find a surprising move and investigate!

### 🧩 Build something tiny

Train a two-layer transformer to substitute `A → C, B → D, C → A, D → B`. Can it apply the rule to unseen strings? *Project idea; no ready-made checkpoint.*

Download this notebook file (File -> Download -> as .ipynb) and then drop it into Claude or ChatGPT and talk to it about what you want to explore!

</details>
